In [1]:
pip install pandas scikit-learn


In [12]:
pip install networkx

In [13]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
from collections import defaultdict
import networkx as nx

def load_data(file_path):
    return pd.read_csv(file_path)

def impute_missing_values(data):
    missing_data = data[data['Latitude'].isna() | data['Longitude'].isna()]
    complete_data = data.dropna(subset=['Latitude', 'Longitude'])
    duplicate_ids = complete_data[complete_data['id'].isin(missing_data['id'])]

    for index, row in missing_data.iterrows():
        if not duplicate_ids[duplicate_ids['id'] == row['id']].empty:
            correct_row = duplicate_ids[duplicate_ids['id'] == row['id']].iloc[0]
            data.loc[index, 'Latitude'] = correct_row['Latitude']
            data.loc[index, 'Longitude'] = correct_row['Longitude']
    return data

def find_neighbors(data):
    coords = data[['Latitude', 'Longitude']].to_numpy()
    coords_rad = np.radians(coords)
    tree = BallTree(coords_rad, metric='haversine')
    radius = 5 / 111  # approximately in radians
    indices = tree.query_radius(coords_rad, r=radius)
    return indices

def create_clusters(data, indices):
    graph = nx.Graph()
    graph.add_nodes_from(data['id'])
    for idx, neighbors in enumerate(indices):
        id_main = data.iloc[idx]['id']
        neighbors_ids = data.iloc[neighbors]['id']
        for neighbor_id in neighbors_ids:
            if id_main != neighbor_id:
                graph.add_edge(id_main, neighbor_id)
    # Find connected components, each component is a cluster
    clusters = list(nx.connected_components(graph))
    cluster_labels = {}
    for i, cluster in enumerate(clusters):
        for id in cluster:
            cluster_labels[id] = f'cluster_{i}'
    return cluster_labels

def assign_cluster_labels(data, cluster_labels):
    data['Cluster'] = data['id'].apply(lambda x: cluster_labels.get(x, 'No cluster'))
    return data

def main():
    file_path = '/content/neibours.csv'  # Adjust this path to your CSV file
    data = load_data(file_path)
    data = impute_missing_values(data)
    indices = find_neighbors(data)
    cluster_labels = create_clusters(data, indices)
    data = assign_cluster_labels(data, cluster_labels)

    # Optionally, save to a new CSV file
    data.to_csv('/content/clustered_neighbors.csv', index=False)
    print("Clustered neighbors data saved to /content/clustered_neighbors.csv")
    print(data.head())

if __name__ == "__main__":
    main()

ValueError: Input contains NaN.